In [54]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Версия Python:", sys.version)
print("Путь к интерпретатору:", sys.executable)
print('numpy :', np.__version__)
print('pandas:', pd.__version__)
print('seaborn:', sns.__version__)

Версия Python: 3.12.16 | packaged by conda-forge | (main, Oct 26 2026, 11:29:01) [MSC v.1944 64 bit (AMD64)]
Путь к интерпретатору: C:\Users\6anna\miniconda3\envs\ds-lab1\python.exe
numpy : 2.5.4
pandas: 3.1.0
seaborn: 0.13.2


In [55]:
# Проверка наличия данных
from pathlib import Path

DATA_DIR = Path("data/raw")

CLIENTS_PATH = DATA_DIR / "clients.csv"
TRANSACTIONS_PATH = DATA_DIR / "transactions.csv"

print("Папка:", DATA_DIR.absolute())
print("clients.csv:", CLIENTS_PATH.is_file())
print("transactions.csv:", TRANSACTIONS_PATH.is_file())

Папка: C:\Users\6anna\PycharmProjects\Suhodolova_Anna_Lab1\data\raw
clients.csv: True
transactions.csv: True


## 1. Аудит качества данных

## 1.1 Наивная загрузка данных

In [56]:
if CLIENTS_PATH.is_file() and TRANSACTIONS_PATH.is_file():
    df_clients_naive = pd.read_csv(CLIENTS_PATH, encoding='utf-8')
    df_transactions_naive = pd.read_csv(TRANSACTIONS_PATH, encoding='cp1251')
    print("\nclients.shape:", df_clients_naive.shape)
    print("transactions.shape:", df_transactions_naive.shape)
    
    print(df_clients_naive.dtypes.value_counts())
    print(df_transactions_naive.dtypes.value_counts())
    
    print(df_clients_naive.head(3))
    print(df_transactions_naive.head(5))
else:
    print("Files not found")
    


clients.shape: (5694, 16)
transactions.shape: (612403, 1)
str        13
float64     2
int64       1
Name: count, dtype: int64
str    1
Name: count, dtype: int64
  client_id  birth_date gender    city      segment loyalty_tier    card_type  \
0   C000897  1971-10-03      Ж  Москва     Массовый      Базовый    Дебетовая   
1   C005884  1959-04-03      Ж  Казань  Премиальный       Золото  Премиальная   
2   C004667  1952-02-05      М  Москва   Зарплатный      Базовый    Дебетовая   

  registration_date card_open_date           income_source  monthly_income  \
0        06.04.2020     04.04.2021               Не указан        103700.0   
1        06.12.2016     13.06.2018  Зарплата в нашем банке        110700.0   
2        28.04.2015     09.04.2016  Зарплата в нашем банке        123700.0   

  has_mobile_app app_last_login  satisfaction_score  branch_code  \
0             Да        2024-04                 2.0          415   
1             Да        2025-01                 5.0          580

Вывод: 
- clients.csv весь прочитан в таблицу (5694, 16)
- transactions.csv прочитан некорректно: получен один столбец вместо отдельных полей, видно, что разделителем является ';'

In [57]:
# Типы всех столбцов
print("\n Типы данных всех столбцов:")
print(df_clients_naive.dtypes)


 Типы данных всех столбцов:
client_id                 str
birth_date                str
gender                    str
city                      str
segment                   str
loyalty_tier              str
card_type                 str
registration_date         str
card_open_date            str
income_source             str
monthly_income        float64
has_mobile_app            str
app_last_login            str
satisfaction_score    float64
branch_code             int64
marketing_opt_in          str
dtype: object


Вывод: - даты(birth_date, registration_date, card_open_date, app_last_login) представлены str, а не спец типом

In [58]:
# Количество пропусков (NaN) в каждом столбце
print("\n Пропуски (NaN) в каждом столбце:")
print(df_clients_naive.isnull().sum())


 Пропуски (NaN) в каждом столбце:
client_id                0
birth_date               0
gender                   0
city                     0
segment                  0
loyalty_tier             0
card_type                0
registration_date        0
card_open_date           0
income_source            0
monthly_income         493
has_mobile_app           0
app_last_login        2145
satisfaction_score    3452
branch_code              0
marketing_opt_in         0
dtype: int64


## 1.2 Правильная загрузка


In [59]:
df_clients = pd.read_csv(
    CLIENTS_PATH,
    encoding='utf-8',
    parse_dates=['birth_date', 'registration_date', 'card_open_date', 'app_last_login'],
    dtype={
        'client_id': 'str',
        'city': 'str',
        'gender': 'category',
        'segment': 'category',
        'loyalty_tier': 'category',
        'card_type': 'category',
        'income_source': 'category',
        'has_mobile_app': 'category',
        'marketing_opt_in': 'category',
        'branch_code': 'int64',
        'satisfaction_score': 'Int64',    #Это порядковая шкала, мне удобно Int так как порядок тогда уже задан 
        'monthly_income': 'float64',
    }
)

df_transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    encoding='cp1251',
    sep=';',
    decimal=',',
    parse_dates=['txn_datetime'],
    dtype={
        'txn_id': 'str',
        'client_id': 'str',
        'currency': 'category',
        'category': 'category',
        'mcc_code': 'str',
        'channel': 'category',
        'status': 'category',
        'amount': 'float64',
    }
)

print("clients.shape:", df_clients.shape)
print("transactions.shape:", df_transactions.shape)

clients.shape: (5694, 16)
transactions.shape: (612403, 9)


## 1.3 Нормализуем df_clients
## Нормализуем категориальные данные

In [60]:
# Список столбцов для анализа
cat_cols = [
    'gender', 'city', 'segment', 'loyalty_tier', 'card_type',
    'income_source', 'has_mobile_app', 'satisfaction_score', 'marketing_opt_in'
]

# Частота категорий в каждом столбце
for col in cat_cols:
    print(f"\n Столбец: {col}")
    print(df_clients[col].value_counts(dropna=False))  # dropna=False — показывает NaN тоже


 Столбец: gender
gender
Ж    2879
М    2815
Name: count, dtype: int64

 Столбец: city
city
Москва                1091
Нижний Новгород        719
Санкт-Петербург        639
Казань                 564
Новосибирск            524
Екатеринбург           481
Н. Новгород            247
г. Москва              207
 москва                188
МОСКВА                 186
казань                 161
г. Новосибирск         141
Санкт Петербург        123
г. Санкт-Петербург     114
ЕКАТЕРИНБУРГ           111
СПб                    107
г.Екатеринбург          91
Name: count, dtype: int64

 Столбец: segment
segment
Массовый       2877
Зарплатный     1665
Премиальный     918
Private         234
Name: count, dtype: int64

 Столбец: loyalty_tier
loyalty_tier
Базовый    2901
Серебро    1527
Золото      967
Платина     299
Name: count, dtype: int64

 Столбец: card_type
card_type
Дебетовая      3910
Кредитная      1295
Премиальная     489
Name: count, dtype: int64

 Столбец: income_source
income_source
Зарплат

Выводы: 
- Есть разные написания одних и тех же городов
- Нет мобильного приложения(has_mobile_app) у 2145 и у 2145 клиентов NaT в app_last_login, надо проверить одни и те же ли это клиенты, но логично что люди у которых нет приложения ни разу в него не заходили
- 1567 клиентов не указали источник зарплаты. Нужно посмотреть опять же связи с другими признаками
- 692 клиента отказались от участия маркетинговой коммуникации, поэтому они не будут включены ни в одну из групп, так как они не интересны
- 3452 пропуска в satisfaction_score нельзя ничем заменить. Далее проверим есть ли связи с другими признаками

In [64]:
# Приведём все города к одному написанию
import re

def normalize_city(city):
    """Brings the city to its standard appearance"""
    if pd.isna(city):
        return city
    
    city = city.strip()  # убираем пробелы
    # Убираем "г. " или "город " в начале
    city = re.sub(r'^(г\.|город)\s*', '', city, flags=re.IGNORECASE)
    city = city.title() # Один регистр
    
    abbreviations = {
        'Спб': 'Санкт-Петербург',
        'Н.Новгород': 'Нижний Новгород',
        'Н. Новгород': 'Нижний Новгород',
        'Ростов-На-Дону': 'Ростов-на-Дону',
    }
    
    return abbreviations.get(city, city)

df_clients['city'] = df_clients['city'].apply(normalize_city)

print(df_clients['city'].value_counts(dropna=False))

city
Москва             1672
Санкт-Петербург     983
Нижний Новгород     966
Казань              725
Екатеринбург        683
Новосибирск         665
Name: count, dtype: int64


In [66]:
# Проверяет соответствие отсутствия даты последнего входа и отсутствие мобильного приложения 
mask_anomaly = (
    (df_clients['has_mobile_app'] == 'Нет') & (df_clients['app_last_login'].notna())  
    |
    (df_clients['has_mobile_app'] == 'Да') & (df_clients['app_last_login'].isna())    
)

df_anomalies = df_clients[mask_anomaly].copy()

print(f"Найдено аномальных записей: {len(df_anomalies)}")
print(f"Из {len(df_clients)} всего ({len(df_anomalies)/len(df_clients)*100:.2f}%)")

# Смотрим на аномалии
display(df_anomalies[['client_id', 'has_mobile_app', 'app_last_login']].head(20))

Найдено аномальных записей: 0
Из 5694 всего (0.00%)


,client_id,has_mobile_app,app_last_login


Вывод: даты последнего входа не указаны только у людей, у которых нет приложения